In [1]:
import pandas as pd

df_results = pd.read_csv("data/results.csv")
df_goalscorers = pd.read_csv("data/goalscorers.csv")
df_shootouts = pd.read_csv("data/shootouts.csv")
df_former_names = pd.read_csv("data/former_names.csv")

print(f"Loaded {len(df_results):,} results, {len(df_goalscorers):,} goalscorers, {len(df_shootouts):,} shootouts")
df_results.head()

Loaded 49,477 results, 47,690 goalscorers, 678 shootouts


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,False
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,False
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,Glasgow,Scotland,False
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,London,England,False
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,Glasgow,Scotland,False


In [2]:
print("filtering datasets...")

filtering datasets...


In [3]:
competitive_tournaments = [
    'FIFA World Cup',
    'FIFA World Cup qualification',
    'UEFA Euro',
    'UEFA Euro qualification',
    'Copa Am\u00e9rica',
    'Copa Am\u00e9rica qualification',
]

In [4]:
df_filtered = df_results[df_results['tournament'].isin(competitive_tournaments)].copy()
df_filtered = df_filtered.sort_values(by='date').reset_index(drop=True)

print(f"Filtering complete! Kept {len(df_filtered):,} competitive matches out of {len(df_results):,} total.")
df_filtered.head()

Filtering complete! Kept 13,896 competitive matches out of 49,477 total.


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
0,1916-07-02,Chile,Uruguay,0.0,4.0,Copa América,Buenos Aires,Argentina,True
1,1916-07-06,Argentina,Chile,6.0,1.0,Copa América,Buenos Aires,Argentina,False
2,1916-07-08,Brazil,Chile,1.0,1.0,Copa América,Buenos Aires,Argentina,True
3,1916-07-10,Argentina,Brazil,1.0,1.0,Copa América,Buenos Aires,Argentina,False
4,1916-07-12,Brazil,Uruguay,1.0,2.0,Copa América,Buenos Aires,Argentina,True


In [8]:
print("Creating target variables...")

#Full time winners only
def get_regular_winners(row):
    if row['home_score'] > row['away_score']:
        return row['home_team']
    elif row['away_score'] > row['home_score']:
        return row['away_team']
    else:
        return 'Draw'


df_filtered['winner'] = df_filtered.apply(get_regular_winners, axis=1)

# 2. Merge shootout winners for matches that ended as a Draw
# We match them up by date, home_team, and away_team
df_merged = pd.merge(
    df_filtered, 
    df_shootouts[['date', 'home_team', 'away_team', 'winner']], 
    on=['date', 'home_team', 'away_team'], 
    how='left', 
    suffixes=('', '_shootout')
)

# 3. If there was a shootout winner, override the 'Draw' status
df_merged['winner'] = df_merged['winner_shootout'].fillna(df_merged['winner'])

# 4. Clean up the temporary shootout column
df_clean = df_merged.drop(columns=['winner_shootout'])

print("✅ Winner column successfully created and adjusted for penalty shootouts!")
df_clean[['date', 'home_team', 'away_team', 'home_score', 'away_score', 'winner']].head(10)

Creating target variables...
✅ Winner column successfully created and adjusted for penalty shootouts!


,date,home_team,away_team,home_score,away_score,winner
0,1916-07-02,Chile,Uruguay,0.0,4.0,Uruguay
1,1916-07-06,Argentina,Chile,6.0,1.0,Argentina
2,1916-07-08,Brazil,Chile,1.0,1.0,Draw
3,1916-07-10,Argentina,Brazil,1.0,1.0,Draw
4,1916-07-12,Brazil,Uruguay,1.0,2.0,Uruguay
5,1916-07-17,Argentina,Uruguay,0.0,0.0,Draw
6,1917-09-30,Uruguay,Chile,4.0,0.0,Uruguay
7,1917-10-03,Argentina,Brazil,4.0,2.0,Argentina
8,1917-10-06,Argentina,Chile,1.0,0.0,Argentina
9,1917-10-07,Uruguay,Brazil,4.0,0.0,Uruguay


In [9]:
# Cell 4: Calculating Rolling Recent Form
print("📈 Calculating team form metrics...")

# 1. Create a helper function to determine points earned by home and away teams
def calculate_points(row):
    if row['winner'] == 'Draw':
        return 1, 1
    elif row['winner'] == row['home_team']:
        return 3, 0
    else:
        return 0, 3

# Apply the function to get points for every match
points = df_clean.apply(calculate_points, axis=1)
df_clean['home_points'] = [p[0] for p in points]
df_clean['away_points'] = [p[1] for p in points]

# 2. Re-organize data to calculate a timeline of points for every single country
team_matches = []
for index, row in df_clean.iterrows():
    team_matches.append({'date': row['date'], 'team': row['home_team'], 'points': row['home_points']})
    team_matches.append({'date': row['date'], 'team': row['away_team'], 'points': row['away_points']})

df_teams = pd.DataFrame(team_matches).sort_values(by=['team', 'date'])

# 3. Calculate the rolling average points over the last 5 matches (excluding the current match)
df_teams['form_last_5'] = df_teams.groupby('team')['points'].transform(
    lambda x: x.shift(1).rolling(5, min_periods=1).mean()
)

# 4. Map these form metrics back into our main match dataset
df_clean = pd.merge(df_clean, df_teams[['date', 'team', 'form_last_5']], 
                    left_on=['date', 'home_team'], right_on=['date', 'team'], how='left').rename(columns={'form_last_5': 'home_form'})
df_clean.drop(columns=['team'], inplace=True)

df_clean = pd.merge(df_clean, df_teams[['date', 'team', 'form_last_5']], 
                    left_on=['date', 'away_team'], right_on=['date', 'team'], how='left').rename(columns={'form_last_5': 'away_form'})
df_clean.drop(columns=['team'], inplace=True)

# Fill initial matches (where a team hasn't played 5 games yet) with the average baseline
df_clean['home_form'] = df_clean['home_form'].fillna(1.0)
df_clean['away_form'] = df_clean['away_form'].fillna(1.0)

print("✅ Team form metrics successfully engineered!")
df_clean[['date', 'home_team', 'away_team', 'home_form', 'away_form', 'winner']].tail(10)

📈 Calculating team form metrics...
✅ Team form metrics successfully engineered!


,date,home_team,away_team,home_form,away_form,winner
13886,2026-06-26,Norway,France,3.0,2.6,Draw
13887,2026-06-26,Egypt,Iran,2.2,1.2,Draw
13888,2026-06-26,New Zealand,Belgium,2.0,1.8,Draw
13889,2026-06-26,Senegal,Iraq,1.8,1.4,Draw
13890,2026-06-27,DR Congo,Uzbekistan,2.2,1.2,Draw
13891,2026-06-27,Colombia,Portugal,2.2,1.2,Draw
13892,2026-06-27,Panama,England,1.6,2.6,Draw
13893,2026-06-27,Algeria,Austria,2.0,1.4,Draw
13894,2026-06-27,Jordan,Argentina,0.8,2.0,Draw
13895,2026-06-27,Croatia,Ghana,2.0,2.6,Draw
